# DuckGPT — 03. Block shapes

> Companion notebook to [`core/04_transformer_block.py`](../core/04_transformer_block.py)
> and [`core/05_model.py`](../core/05_model.py).

Forward a small fake batch through a single transformer block and through a
two-layer GPT, printing every shape so you can keep the dimensions in your
head.

In [ ]:
import sys, torch
sys.path.append('../core')
from importlib import import_module
m = import_module('05_model')
GPT, GPTConfig = m.GPT, m.GPTConfig

cfg = GPTConfig(vocab_size=128, block_size=16, n_layer=2, n_head=4, n_embd=64)
model = GPT(cfg)
print('params:', f'{sum(p.numel() for p in model.parameters()):,}')

In [ ]:
torch.manual_seed(0)
idx = torch.randint(0, cfg.vocab_size, (2, cfg.block_size))
print('input ids:', idx.shape)
x = model.wte(idx) + model.wpe(torch.arange(cfg.block_size))
print('after embedding:', x.shape, ' (B, T, n_embd)')
for i, blk in enumerate(model.blocks):
    x = blk(x)
    print(f'after block {i}:', x.shape)
x = model.ln_f(x)
logits = model.lm_head(x)
print('logits:', logits.shape, ' (B, T, vocab_size)')

## Takeaway

Inside the model the tensor stays at shape `(B, T, n_embd)` until the final
linear layer projects it to vocabulary logits. Each block consumes and
returns the same shape — that's what lets you stack as many as you can
afford to train.